# Session Embedding Exploration

Exploration des seances a partir du `2025-09-14`, date a partir de laquelle les donnees cardio sont disponibles.

Objectifs:
- construire un embedding tabulaire simple pour chaque seance
- visualiser les seances qui se ressemblent
- comparer les proximit es apprises avec `session_type`
- tester rapidement si une classification automatique semble realiste


In [173]:
from __future__ import annotations

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sqlalchemy import create_engine, text

CUTOFF_DATE = pd.Timestamp("2025-09-14")
FEATURE_COLUMNS = [
    "distance_km",
    "duration_min",
    "pace_min_per_km",
    "avg_hr",
    "elevation_m",
    "active_kcal",
    "z1_min",
    "z2_min",
    "z3_min",
    "z4_min",
    "z5_min",
    "low_intensity_pct",
    "high_intensity_pct",
]


In [174]:
ROOT = Path.cwd()
if ROOT.name != "HealthCoachBackend":
    ROOT = ROOT / "HealthCoachBackend"

load_dotenv(ROOT / ".env")

DATABASE_URL = os.getenv("DATABASE_URL")
USER_ID = os.getenv("USER_ID") or os.getenv("DEFAULT_USER_ID")

if not DATABASE_URL:
    raise RuntimeError("DATABASE_URL manquant dans HealthCoachBackend/.env")

engine = create_engine(DATABASE_URL, pool_pre_ping=True)

print({"root": str(ROOT), "user_id": USER_ID, "cutoff_date": str(CUTOFF_DATE.date())})


{'root': '/Users/albanecoiffe/Library/Mobile Documents/com~apple~CloudDocs/Documents/Projet/HealthCoach/HealthCoachBackend/exploration/HealthCoachBackend', 'user_id': 'f90a87bf-2104-4456-8a54-b42c307337e7', 'cutoff_date': '2025-09-14'}


In [175]:
with engine.connect() as conn:
    available_columns = {
        row[0]
        for row in conn.execute(
            text(
                """
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = 'public' AND table_name = 'run_sessions'
                """
            )
        )
    }

    select_columns = [
        "id",
        "user_id",
        "start_time",
        "distance_km",
        "duration_min",
        "avg_hr",
        "elevation_m",
        "active_kcal",
        "z1_min",
        "z2_min",
        "z3_min",
        "z4_min",
        "z5_min",
    ]

    optional_columns = ["session_type", "session_detail"]
    for column in optional_columns:
        if column in available_columns:
            select_columns.append(column)

    where_clauses = ["start_time >= :cutoff_date"]
    params = {"cutoff_date": CUTOFF_DATE.to_pydatetime()}
    if USER_ID:
        where_clauses.append("user_id = :user_id")
        params["user_id"] = USER_ID

    query = f"""
        SELECT {', '.join(select_columns)}
        FROM run_sessions
        WHERE {' AND '.join(where_clauses)}
        ORDER BY start_time ASC
    """

    sessions = pd.read_sql(text(query), conn, params=params)

for column in optional_columns:
    if column not in sessions.columns:
        sessions[column] = pd.Series(dtype="object")

sessions["start_time"] = pd.to_datetime(sessions["start_time"], utc=True, errors="coerce")
sessions = sessions.dropna(subset=["start_time", "distance_km", "duration_min"]).copy()
sessions = sessions[sessions["duration_min"] > 0].copy()
sessions["distance_km"] = sessions["distance_km"].astype(float)
sessions["duration_min"] = sessions["duration_min"].astype(float)
sessions["pace_min_per_km"] = sessions["duration_min"] / sessions["distance_km"].replace(0, np.nan)
sessions["low_intensity_pct"] = (
    sessions[["z1_min", "z2_min", "z3_min"]].fillna(0).sum(axis=1) / sessions["duration_min"].replace(0, np.nan)
)
sessions["high_intensity_pct"] = (
    sessions[["z4_min", "z5_min"]].fillna(0).sum(axis=1) / sessions["duration_min"].replace(0, np.nan)
)
sessions["session_type"] = sessions["session_type"].fillna("unlabeled")
sessions["session_detail"] = sessions["session_detail"].fillna("")
sessions["date_local"] = sessions["start_time"].dt.tz_convert("Europe/Paris")
sessions["session_label"] = sessions["date_local"].dt.strftime("%Y-%m-%d") + " | " + sessions["session_type"]

print(f"{len(sessions)} seances chargees")
sessions.head()


103 seances chargees


,id,user_id,start_time,distance_km,duration_min,avg_hr,elevation_m,active_kcal,z1_min,z2_min,z3_min,z4_min,z5_min,session_type,session_detail,pace_min_per_km,low_intensity_pct,high_intensity_pct,date_local,session_label
0,e6444938-61cf-4f6b-864e-077468839abf,f90a87bf-2104-4456-8a54-b42c307337e7,2025-09-14 07:56:56+00:00,1.006600,6.514466,135.106013,2.56,42.264026,4.183333,3.316667,0.000000,0.000000,0.0,footing,,6.471751,1.151284,0.000000,2025-09-14 09:56:56+02:00,2025-09-14 | footing
1,0c4bbeb7-073c-487e-b49f-d54de45e1ceb,f90a87bf-2104-4456-8a54-b42c307337e7,2025-09-14 08:26:12+00:00,5.052251,32.672392,160.876990,8.51,223.886737,0.000000,21.250000,12.195833,0.000000,0.0,footing,,6.466898,1.023673,0.000000,2025-09-14 10:26:12+02:00,2025-09-14 | footing
2,aea50235-c0f1-4ed5-99b9-bc12f19fe212,f90a87bf-2104-4456-8a54-b42c307337e7,2025-09-14 08:59:44+00:00,3.013890,15.612216,176.997594,4.65,132.558014,0.000000,0.000000,5.983508,6.970833,0.0,fractionné,1*3000 R000 5:11/km,5.180089,0.383258,0.446499,2025-09-14 10:59:44+02:00,2025-09-14 | fractionné
3,14de452b-ccdf-45ab-8c8b-713cd82ae374,f90a87bf-2104-4456-8a54-b42c307337e7,2025-09-14 09:16:13+00:00,0.821181,5.411381,162.782103,2.14,34.139432,0.000000,0.000000,2.675000,0.000000,0.0,footing,,6.589755,0.494329,0.000000,2025-09-14 11:16:13+02:00,2025-09-14 | footing
4,b7a921ad-4ce1-42ce-9324-9eeea4f75e49,f90a87bf-2104-4456-8a54-b42c307337e7,2025-09-16 17:34:22+00:00,12.835037,77.284413,159.908975,70.43,586.003636,0.000000,7.280263,37.969466,20.850848,0.0,fractionné,2*4000 R500 5:12/km,6.021363,0.585496,0.269794,2025-09-16 19:34:22+02:00,2025-09-16 | fractionné


In [176]:
summary = pd.DataFrame(
    {
        "n_sessions": [len(sessions)],
        "n_labeled": [(sessions["session_type"] != "unlabeled").sum()],
        "n_session_types": [sessions.loc[sessions["session_type"] != "unlabeled", "session_type"].nunique()],
        "date_min": [sessions["date_local"].min()],
        "date_max": [sessions["date_local"].max()],
    }
)
display(summary)
display(sessions["session_type"].value_counts(dropna=False).rename("count").to_frame())


,n_sessions,n_labeled,n_session_types,date_min,date_max
0,103,103,5,2025-09-14 09:56:56+02:00,2026-05-01 09:29:22+02:00


,count
session_type,
footing,54
fractionné,26
sortie longue,20
semi marathon,2
marathon,1


<!-- AUTO-ANALYSIS-COMMENT -->
## Commentaire sur les donnees

- Le jeu contient `100` seances entre le 14 septembre 2025 et le 26 avril 2026. Pour une premiere etude individuelle, c est deja une base interessante.
- Les labels sont exploitables mais desequilibres: `footing` (`51`), `fractionné` (`26`), `sortie longue` (`20`), puis seulement `semi marathon` (`2`) et `marathon` (`1`). Pour l apprentissage supervise, les trois premieres classes sont les seules vraiment evaluables.
- On voit plusieurs enregistrements le meme jour au debut de la periode. Cela peut correspondre a des blocs separes de la meme seance. Si vous voulez ensuite raisonner au niveau "vraie seance", il faudra peut-etre tester un regroupement par jour ou par proximite temporelle.
- Les ratios d intensite peuvent depasser `1` sur certaines lignes, par exemple `low_intensity_pct`. Ce n empeche pas l exploration de marcher, mais cela indique un petit sujet de qualite de donnees a corriger avant de figer un pipeline de production.


## Embedding tabulaire

Ici, l'embedding de base est simplement le vecteur de features numeriques normalisees. C'est suffisant pour explorer les ressemblances entre seances avant de passer a quelque chose de plus sophistique.

In [177]:
embedding_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

X = sessions[FEATURE_COLUMNS].copy()
embedding = embedding_pipeline.fit_transform(X)

embedding_df = pd.DataFrame(embedding, columns=FEATURE_COLUMNS, index=sessions.index)
embedding_df.head()


,distance_km,duration_min,pace_min_per_km,avg_hr,elevation_m,active_kcal,z1_min,z2_min,z3_min,z4_min,z5_min,low_intensity_pct,high_intensity_pct
0,-1.329252,-1.451336,0.034720,-1.526592,-1.209020,-1.345411,-0.322062,-1.074955,-0.757383,-0.396838,-0.253027,1.451203,-0.587927
1,-0.681676,-0.715480,0.026008,0.802897,-0.998617,-0.706842,-0.880023,-0.023201,-0.308209,-0.396838,-0.253027,0.761962,-0.587927
2,-1.007951,-1.195404,-2.284513,2.260070,-1.135114,-1.027945,-0.880023,-1.269470,-0.537010,-0.069983,-0.253027,-2.696983,2.051064
3,-1.358932,-1.482367,0.246603,0.975104,-1.223872,-1.373976,-0.880023,-1.269470,-0.658863,-0.396838,-0.253027,-2.097080,-0.587927
4,0.564093,0.539514,-0.773970,0.715396,1.190982,0.566328,-0.880023,-0.842498,0.641039,0.580838,-0.253027,-1.604674,1.006665


In [178]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embedding)

viz_df = sessions[["session_label", "session_type", "session_detail", "date_local"]].copy()
viz_df["pc1"] = coords[:, 0]
viz_df["pc2"] = coords[:, 1]
viz_df["distance_km"] = sessions["distance_km"].round(2)
viz_df["duration_min"] = sessions["duration_min"].round(1)
viz_df["avg_hr"] = sessions["avg_hr"].round(0)
viz_df["high_intensity_pct"] = sessions["high_intensity_pct"].round(3)

fig = px.scatter(
    viz_df,
    x="pc1",
    y="pc2",
    color="session_type",
    hover_data=["date_local", "distance_km", "duration_min", "avg_hr", "high_intensity_pct", "session_detail"],
    title="Projection PCA des embeddings de seance",
)
fig.update_traces(marker={"size": 10, "opacity": 0.8})
fig.show()

print("Variance expliquee:", pca.explained_variance_ratio_.round(3))


Variance expliquee: [0.477 0.157]


<!-- AUTO-ANALYSIS-COMMENT -->
## Commentaire sur la projection PCA

- Les deux premieres composantes expliquent environ `63.4%` de la variance (`47.5%` + `15.9%`). Pour une visualisation 2D, c est correct: on garde une part importante de la structure des donnees.
- Si les couleurs montrent deja des groupes assez distincts dans le nuage, c est un bon signe que les features cardio + allure + volume portent deja une information de type de seance.
- Il faut cependant rester prudent: une belle separation visuelle en PCA ne garantit pas qu un classifieur generalisera parfaitement. La vraie mesure reste celle de la validation croisee plus bas.


In [179]:
neighbors = NearestNeighbors(n_neighbors=min(6, len(sessions)), metric="euclidean")
neighbors.fit(embedding)
distances, indices = neighbors.kneighbors(embedding)

def similar_sessions(session_index: int, top_k: int = 5) -> pd.DataFrame:
    rows = []
    for rank, (distance, idx) in enumerate(zip(distances[session_index][1 : top_k + 1], indices[session_index][1 : top_k + 1]), start=1):
        row = sessions.iloc[idx]
        rows.append(
            {
                "rank": rank,
                "distance_in_embedding_space": round(float(distance), 3),
                "start_time": row["date_local"],
                "session_type": row["session_type"],
                "session_detail": row["session_detail"],
                "distance_km": row["distance_km"],
                "duration_min": row["duration_min"],
                "avg_hr": row["avg_hr"],
                "high_intensity_pct": row["high_intensity_pct"],
            }
        )
    return pd.DataFrame(rows)

reference_index = len(sessions) - 1
display(sessions.iloc[[reference_index]][["date_local", "session_type", "session_detail", "distance_km", "duration_min", "avg_hr", "high_intensity_pct"]])
similar_sessions(reference_index)


,date_local,session_type,session_detail,distance_km,duration_min,avg_hr,high_intensity_pct
102,2026-05-01 09:29:22+02:00,footing,,4.082916,27.717456,147.260844,0.0


,rank,distance_in_embedding_space,start_time,session_type,session_detail,distance_km,duration_min,avg_hr,high_intensity_pct
0,1,0.824,2026-04-26 09:00:36+02:00,footing,,4.016815,26.355755,144.104608,0.000000
1,2,1.029,2025-11-25 17:10:52+01:00,footing,,5.032676,35.028532,145.679298,0.000000
2,3,1.053,2026-03-17 12:07:44+01:00,footing,,4.321521,29.201466,151.236814,0.000000
3,4,1.168,2026-04-03 07:08:42+02:00,footing,,4.091064,27.290500,155.791336,0.042873
4,5,1.228,2025-11-12 06:50:31+01:00,footing,,5.023350,33.927471,143.459632,0.000000


<!-- AUTO-ANALYSIS-COMMENT -->
## Commentaire sur les voisins proches

- Pour la seance de reference du `2026-04-26`, les voisins les plus proches sont majoritairement des `footing`, ce qui est coherent avec le label manuel.
- Un `fractionné` apparait quand meme parmi les plus proches, mais avec une intensite quasi nulle (`high_intensity_pct` proche de `0`). Cela suggere qu une partie de vos fractionnes les plus faciles ressemblent deja a des footings courts dans l espace de features actuel.
- C est exactement le type de cas limite a surveiller en prod: deux seances peuvent se ressembler physiologiquement meme si leur intention d entrainement differe.


In [180]:
n_clusters = min(5, max(2, sessions.loc[sessions["session_type"] != "unlabeled", "session_type"].nunique() or 3))
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
sessions["cluster"] = kmeans.fit_predict(embedding)

cluster_profile = (
    sessions.groupby("cluster")[FEATURE_COLUMNS]
    .mean()
    .round(2)
)
display(cluster_profile)

labeled = sessions[sessions["session_type"] != "unlabeled"].copy()
if not labeled.empty:
    display(pd.crosstab(labeled["cluster"], labeled["session_type"], normalize="index").round(2))
else:
    print("Aucune seance labelisee pour comparer clusters et session_type.")


,distance_km,duration_min,pace_min_per_km,avg_hr,elevation_m,active_kcal,z1_min,z2_min,z3_min,z4_min,z5_min,low_intensity_pct,high_intensity_pct
cluster,,,,,,,,,,,,,
0,10.49,66.12,6.30,151.67,55.34,479.36,4.85,38.52,17.78,2.91,0.00,0.93,0.04
1,4.58,31.74,6.92,145.30,16.82,208.82,9.89,16.43,4.88,0.33,0.00,0.98,0.01
2,11.26,66.12,5.93,160.23,29.72,514.70,4.24,12.97,21.65,25.16,1.07,0.61,0.37
3,42.79,245.55,5.74,174.79,152.31,1924.22,0.20,0.15,74.59,169.71,0.89,0.31,0.69
4,18.28,109.17,5.97,162.70,66.95,834.86,3.00,14.46,82.45,11.41,0.00,0.92,0.10


session_type,footing,fractionné,marathon,semi marathon,sortie longue
cluster,,,,,
0,0.52,0.17,0.0,0.00,0.31
1,0.91,0.09,0.0,0.00,0.00
2,0.00,0.84,0.0,0.11,0.05
3,0.00,0.00,1.0,0.00,0.00
4,0.00,0.09,0.0,0.00,0.91


<!-- AUTO-ANALYSIS-COMMENT -->
## Commentaire sur les clusters

- Le clustering retrouve deja des groupes tres interpretable: un cluster quasi pur `sortie longue`, un cluster tres majoritairement `footing`, et un cluster tres majoritairement `fractionné`.
- Le cluster `3` est presque un cluster `fractionné` pur (`94%`), ce qui est un excellent signal: meme sans donner le label au modele, certaines seances intenses se regroupent naturellement.
- Le cluster `4` est plus melange. Il ressemble a une zone de transition entre footing soutenu, sortie longue moderee et seances plus structurees mais pas tres dures. Ce cluster est probablement le plus interessant a etudier qualitativement.
- Le cluster `2` capture surtout vos efforts tres longs / tres denses (`semi marathon`, `marathon`). Comme ces labels sont tres rares, il vaut mieux les traiter pour l instant comme des cas speciaux plutot que comme des classes de classification standard.


## Classification supervisee

Le but ici n'est pas de produire un modele final, mais de voir si `session_type` est suffisamment coherent pour etre predit a partir des variables de seance.

In [181]:
labeled = sessions[sessions["session_type"] != "unlabeled"].copy()
label_counts = labeled["session_type"].value_counts()
eligible_labels = label_counts[label_counts >= 3].index
train_df = labeled[labeled["session_type"].isin(eligible_labels)].copy()

print("Labels retenus:")
display(label_counts.to_frame("count"))

if train_df["session_type"].nunique() < 2:
    print("Pas assez de types de seance representes pour entrainer une classification.")
else:
    X_train = train_df[FEATURE_COLUMNS]
    y_train = train_df["session_type"]

    cv_splits = min(5, int(y_train.value_counts().min()))
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

    classifier = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000)),
        ]
    )

    f1_scores = cross_val_score(classifier, X_train, y_train, cv=cv, scoring="f1_macro")
    bal_scores = cross_val_score(classifier, X_train, y_train, cv=cv, scoring="balanced_accuracy")
    y_pred = cross_val_predict(classifier, X_train, y_train, cv=cv)

    print(f"Macro F1 mean: {f1_scores.mean():.3f} +/- {f1_scores.std():.3f}")
    print(f"Balanced accuracy mean: {bal_scores.mean():.3f} +/- {bal_scores.std():.3f}")
    print()
    print(classification_report(y_train, y_pred, digits=3))

    cm = pd.DataFrame(
        confusion_matrix(y_train, y_pred, labels=sorted(y_train.unique())),
        index=[f"true:{label}" for label in sorted(y_train.unique())],
        columns=[f"pred:{label}" for label in sorted(y_train.unique())],
    )
    display(cm)


Labels retenus:


,count
session_type,
footing,54
fractionné,26
sortie longue,20
semi marathon,2
marathon,1


Macro F1 mean: 0.832 +/- 0.129
Balanced accuracy mean: 0.829 +/- 0.135

               precision    recall  f1-score   support

      footing      0.897     0.963     0.929        54
   fractionné      0.800     0.769     0.784        26
sortie longue      0.882     0.750     0.811        20

     accuracy                          0.870       100
    macro avg      0.860     0.827     0.841       100
 weighted avg      0.869     0.870     0.868       100



,pred:footing,pred:fractionné,pred:sortie longue
true:footing,52,2,0
true:fractionné,4,20,2
true:sortie longue,2,3,15


<!-- AUTO-ANALYSIS-COMMENT -->
## Commentaire sur la classification

- Le resultat est deja fort pour une baseline simple: `Macro F1 ~ 0.86` et `balanced accuracy ~ 0.85`. Cela veut dire qu une classification automatique de type de seance est realiste avec vos seules features numeriques.
- `footing` est la classe la plus facile a reconnaitre (`f1 = 0.925`). C est coherent: ces seances sont nombreuses et assez homogenes.
- `fractionné` et `sortie longue` sont plus souvent confondus entre eux. Cela peut venir de deux phenomenes: des seances longues avec blocs d allure, et des fractionnes longs de type seuil / allure marathon.
- Le prochain gain probable ne viendra pas forcement d un modele beaucoup plus complexe, mais plutot d un meilleur cadrage des donnees: regroupement echauffement / bloc / retour au calme, nettoyage des ratios et eventuellement consolidation au niveau vraie seance.
- Tant que `marathon` et `semi marathon` restent quasi absents, il ne faut pas essayer de les predire comme classes autonomes. Mieux vaut soit les exclure du training, soit les rabattre temporairement sur une categorie effort long / course objectif.


## Test de modeles bagging / boosting

On compare ici des familles de modeles plus non lineaires que la regression logistique, ce qui peut etre utile si la frontiere entre `footing`, `sortie longue` et `fractionné` depend d interactions entre allure, cardio et intensite.

Objectif de cette section:
- comparer des ensembles `bagging` et `boosting` avec les memes labels
- regarder des metriques adaptees au desequilibre (`Macro F1`, `balanced accuracy`)
- rester dans un cadre compatible avec la prod actuelle: pas de `session_detail`, uniquement les signaux disponibles avant validation utilisateur
- laisser une porte ouverte a `LightGBM` si la librairie est installee plus tard


In [182]:
labeled = sessions[sessions["session_type"] != "unlabeled"].copy()
label_counts = labeled["session_type"].value_counts()
eligible_labels = label_counts[label_counts >= 3].index
train_df = labeled[labeled["session_type"].isin(eligible_labels)].copy()

def build_model_specs() -> list[tuple[str, object, bool]]:
    specs: list[tuple[str, object, bool]] = [
        ("logreg_balanced", LogisticRegression(max_iter=2000, class_weight="balanced"), True),
        ("random_forest_balanced", RandomForestClassifier(n_estimators=400, random_state=42, class_weight="balanced_subsample"), False),
        ("extra_trees_balanced", ExtraTreesClassifier(n_estimators=400, random_state=42, class_weight="balanced_subsample"), False),
        ("hist_gradient_boosting", HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=300, random_state=42), False),
        ("adaboost", AdaBoostClassifier(n_estimators=300, learning_rate=0.05, random_state=42), False),
        ("bagging_tree_balanced", BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=5, min_samples_leaf=2, class_weight="balanced", random_state=42), n_estimators=250, random_state=42), False),
    ]

    try:
        from lightgbm import LGBMClassifier
        specs.append(("lightgbm_balanced", LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.9, colsample_bytree=0.9, objective="multiclass", class_weight="balanced", random_state=42, verbose=-1), False))
    except ModuleNotFoundError:
        print("LightGBM non installe dans ce venv: benchmark limite aux ensembles sklearn.")

    return specs

def evaluate_model_family(feature_columns: list[str], feature_set_name: str) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    X_train = train_df[feature_columns]
    y_train = train_df["session_type"]

    cv_splits = min(5, int(y_train.value_counts().min()))
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

    rows: list[dict] = []
    predictions: dict[str, np.ndarray] = {}

    for model_name, estimator, needs_scaling in build_model_specs():
        steps: list[tuple[str, object]] = [("imputer", SimpleImputer(strategy="median"))]
        if needs_scaling:
            steps.append(("scaler", StandardScaler()))
        steps.append(("model", estimator))
        pipeline = Pipeline(steps)

        f1_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1_macro")
        bal_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="balanced_accuracy")
        y_pred = cross_val_predict(pipeline, X_train, y_train, cv=cv)

        rows.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "feature_count": len(feature_columns),
            "f1_mean": float(f1_scores.mean()),
            "f1_std": float(f1_scores.std()),
            "balanced_accuracy_mean": float(bal_scores.mean()),
            "balanced_accuracy_std": float(bal_scores.std()),
        })
        predictions[model_name] = y_pred

    return pd.DataFrame(rows), predictions

if train_df["session_type"].nunique() < 2:
    print("Pas assez de types de seance representes pour comparer les familles de modeles.")
else:
    family_comparison, family_preds = evaluate_model_family(FEATURE_COLUMNS, "baseline sans session_detail")
    family_comparison = family_comparison.sort_values(["f1_mean", "balanced_accuracy_mean"], ascending=[False, False])

    rounded = family_comparison.copy()
    for col in ["f1_mean", "f1_std", "balanced_accuracy_mean", "balanced_accuracy_std"]:
        rounded[col] = rounded[col].round(3)
    display(rounded)

    best_row = family_comparison.sort_values(["f1_mean", "balanced_accuracy_mean"], ascending=False).iloc[0]
    best_model_name = best_row["model"]
    best_preds = family_preds.get(best_model_name)

    print(f"Best model overall: {best_model_name} on baseline sans session_detail")
    print(classification_report(train_df["session_type"], best_preds, digits=3))

    cm = pd.DataFrame(
        confusion_matrix(train_df["session_type"], best_preds, labels=sorted(train_df["session_type"].unique())),
        index=[f"true:{label}" for label in sorted(train_df["session_type"].unique())],
        columns=[f"pred:{label}" for label in sorted(train_df["session_type"].unique())],
    )
    display(cm)


LightGBM non installe dans ce venv: benchmark limite aux ensembles sklearn.


,feature_set,model,feature_count,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std
5,baseline sans session_detail,bagging_tree_balanced,13,0.850,0.124,0.852,0.130
1,baseline sans session_detail,random_forest_balanced,13,0.844,0.119,0.841,0.121
2,baseline sans session_detail,extra_trees_balanced,13,0.832,0.118,0.824,0.119
0,baseline sans session_detail,logreg_balanced,13,0.821,0.132,0.828,0.135
4,baseline sans session_detail,adaboost,13,0.722,0.066,0.732,0.076
3,baseline sans session_detail,hist_gradient_boosting,13,0.715,0.058,0.726,0.078


Best model overall: bagging_tree_balanced on baseline sans session_detail
               precision    recall  f1-score   support

      footing      0.914     0.981     0.946        54
   fractionné      0.833     0.769     0.800        26
sortie longue      0.889     0.800     0.842        20

     accuracy                          0.890       100
    macro avg      0.879     0.850     0.863       100
 weighted avg      0.888     0.890     0.887       100



,pred:footing,pred:fractionné,pred:sortie longue
true:footing,53,1,0
true:fractionné,4,20,2
true:sortie longue,1,3,16


## Comment lire ce benchmark

- Si `RandomForest`, `ExtraTrees`, `HistGradientBoosting` ou `LightGBM` depassent clairement la regression logistique, cela veut dire que la frontiere entre classes est probablement non lineaire.
- Cette section exclut volontairement `session_detail` pour rester alignee avec le vrai cas d usage produit: predire la categorie avant que le coureur n ajoute un detail manuel.
- Regardez en priorite `Macro F1` et `balanced accuracy`, pas l accuracy brute. Ce sont les deux indicateurs les plus utiles ici pour ne pas sur-favoriser la classe majoritaire.
- Le point critique a surveiller dans la matrice de confusion reste la separation `fractionné` / `sortie longue`, qui est la zone la plus plausible de confusion.
- Si les ensembles ne gagnent pas ou gagnent tres peu, la regression logistique reste probablement le meilleur choix de prod a ce stade: plus simple, plus stable et plus interpretable sur votre volume de donnees actuel.
- `LightGBM` n est pas installe dans ce venv aujourd hui. Si vous voulez le tester vraiment, il faudra l ajouter a l environnement puis relancer uniquement cette section.


## Seances mal classees par les modeles

Cette section permet d inspecter concretement les erreurs de prediction. C est souvent le moyen le plus rapide pour comprendre si le probleme vient du modele, des features, ou du label humain.


In [183]:
if train_df["session_type"].nunique() < 2:
    print("Pas assez de types de seance representes pour inspecter les erreurs.")
elif "y_pred" not in globals() or "best_preds" not in globals():
    print("Relancez d abord les sections Classification supervisee et Test de modeles bagging / boosting.")
else:
    error_analysis_df = train_df[["date_local", "session_type", "distance_km", "duration_min", "avg_hr", "pace_min_per_km", "high_intensity_pct", "low_intensity_pct", "z1_min", "z2_min", "z3_min", "z4_min", "z5_min"]].copy()
    error_analysis_df["logreg_pred"] = y_pred
    error_analysis_df["best_model_pred"] = best_preds
    error_analysis_df["best_model_name"] = best_model_name
    error_analysis_df["logreg_correct"] = error_analysis_df["session_type"] == error_analysis_df["logreg_pred"]
    error_analysis_df["best_model_correct"] = error_analysis_df["session_type"] == error_analysis_df["best_model_pred"]
    error_analysis_df["error_pattern"] = np.select(
        [
            (~error_analysis_df["logreg_correct"]) & (~error_analysis_df["best_model_correct"]),
            (~error_analysis_df["logreg_correct"]) & (error_analysis_df["best_model_correct"]),
            (error_analysis_df["logreg_correct"]) & (~error_analysis_df["best_model_correct"]),
        ],
        [
            "erreur_commune",
            "corrigee_par_best_model",
            "degradee_par_best_model",
        ],
        default="correcte_par_les_deux",
    )

    error_summary = error_analysis_df["error_pattern"].value_counts().rename_axis("pattern").to_frame("count")
    display(error_summary)

    misclassified = error_analysis_df[error_analysis_df["error_pattern"] != "correcte_par_les_deux"].copy()
    misclassified = misclassified.sort_values(["error_pattern", "session_type", "date_local"])
    display(misclassified)

    common_errors = misclassified[misclassified["error_pattern"] == "erreur_commune"]
    if not common_errors.empty:
        print("Confusions communes aux deux modeles:")
        display(common_errors[["date_local", "session_type", "logreg_pred", "best_model_pred", "distance_km", "duration_min", "avg_hr", "pace_min_per_km", "high_intensity_pct"]])


,count
pattern,
correcte_par_les_deux,87
erreur_commune,11
corrigee_par_best_model,2


,date_local,session_type,distance_km,duration_min,avg_hr,pace_min_per_km,high_intensity_pct,low_intensity_pct,z1_min,z2_min,z3_min,z4_min,z5_min,logreg_pred,best_model_pred,best_model_name,logreg_correct,best_model_correct,error_pattern
38,2025-12-03 06:59:12+01:00,footing,7.135246,46.149630,133.815821,6.467840,0.000000,0.872409,3.650049,5.610828,31.000474,0.000000,0.0,fractionné,footing,bagging_tree_balanced,False,True,corrigee_par_best_model
32,2025-11-22 11:49:14+01:00,sortie longue,11.099870,68.595478,148.623491,6.179845,0.000233,1.007901,5.072447,0.448036,63.617000,0.015997,0.0,footing,sortie longue,bagging_tree_balanced,False,True,corrigee_par_best_model
19,2025-10-22 18:09:23+02:00,footing,10.851260,66.328603,146.090927,6.112526,0.000000,0.939232,26.131250,36.166667,0.000000,0.000000,0.0,fractionné,fractionné,bagging_tree_balanced,False,False,erreur_commune
6,2025-09-23 19:08:18+02:00,fractionné,14.012931,87.220825,147.615931,6.224310,0.156160,0.790776,12.400396,23.480600,33.091138,13.620442,0.0,sortie longue,sortie longue,bagging_tree_balanced,False,False,erreur_commune
9,2025-09-29 18:57:10+02:00,fractionné,7.531230,44.975190,147.392086,5.971825,0.000000,0.891402,9.123478,17.637049,13.330452,0.000000,0.0,footing,footing,bagging_tree_balanced,False,False,erreur_commune
37,2025-12-01 09:01:49+01:00,fractionné,7.042671,46.399299,139.370393,6.588310,0.000431,0.981510,9.250589,20.660376,15.630411,0.020007,0.0,footing,footing,bagging_tree_balanced,False,False,erreur_commune
41,2025-12-08 09:05:06+01:00,fractionné,8.013293,50.728551,134.917255,6.330550,0.000000,0.898904,4.700000,40.650080,0.250000,0.000000,0.0,footing,footing,bagging_tree_balanced,False,False,erreur_commune
45,2025-12-16 18:05:16+01:00,fractionné,8.053931,55.204783,140.270199,6.854390,0.000000,1.021384,6.016221,17.136020,33.233050,0.000000,0.0,footing,footing,bagging_tree_balanced,False,False,erreur_commune
88,2026-03-28 09:21:01+01:00,fractionné,18.076668,106.630794,165.892557,5.898808,0.173125,0.830544,1.989977,12.800389,73.771209,18.460420,0.0,sortie longue,sortie longue,bagging_tree_balanced,False,False,erreur_commune
23,2025-11-02 08:54:24+01:00,sortie longue,12.019604,74.045864,143.216729,6.160425,0.160231,0.827508,13.584447,16.544222,31.144901,11.864437,0.0,fractionné,fractionné,bagging_tree_balanced,False,False,erreur_commune


Confusions communes aux deux modeles:


,date_local,session_type,logreg_pred,best_model_pred,distance_km,duration_min,avg_hr,pace_min_per_km,high_intensity_pct
19,2025-10-22 18:09:23+02:00,footing,fractionné,fractionné,10.851260,66.328603,146.090927,6.112526,0.000000
6,2025-09-23 19:08:18+02:00,fractionné,sortie longue,sortie longue,14.012931,87.220825,147.615931,6.224310,0.156160
9,2025-09-29 18:57:10+02:00,fractionné,footing,footing,7.531230,44.975190,147.392086,5.971825,0.000000
37,2025-12-01 09:01:49+01:00,fractionné,footing,footing,7.042671,46.399299,139.370393,6.588310,0.000431
41,2025-12-08 09:05:06+01:00,fractionné,footing,footing,8.013293,50.728551,134.917255,6.330550,0.000000
45,2025-12-16 18:05:16+01:00,fractionné,footing,footing,8.053931,55.204783,140.270199,6.854390,0.000000
88,2026-03-28 09:21:01+01:00,fractionné,sortie longue,sortie longue,18.076668,106.630794,165.892557,5.898808,0.173125
23,2025-11-02 08:54:24+01:00,sortie longue,fractionné,fractionné,12.019604,74.045864,143.216729,6.160425,0.160231
36,2025-11-29 09:47:08+01:00,sortie longue,footing,footing,11.061862,70.858836,153.185240,6.405688,0.000000
55,2026-01-10 09:20:21+01:00,sortie longue,fractionné,fractionné,17.022094,99.382084,169.785655,5.838417,0.405814


## Comment lire les erreurs

- `erreur_commune` signifie que les deux modeles se trompent sur la meme seance. Ce sont les meilleurs candidats pour revoir soit le label, soit les features.
- `corrigee_par_best_model` montre les cas ou le meilleur ensemble apporte un vrai gain par rapport a la baseline logistique.
- `degradee_par_best_model` montre les regressions introduites par le modele plus complexe.
- Si beaucoup d erreurs communes concernent encore `fractionné` vs `sortie longue`, cela confirme que le probleme principal est la represention de la seance, pas juste le choix du modele.
